## Running a local registry

For testing, air-gapped networks, or as the core of a private registry, the OSS **`registry:2`** image runs a fully-functional OCI distribution server in one container:

```bash
docker run -d -p 5000:5000 --name registry --restart=always registry:2

docker tag flask-demo:0.1 localhost:5000/flask-demo:0.1
docker push localhost:5000/flask-demo:0.1
docker pull localhost:5000/flask-demo:0.1
```

That's a working registry in two commands — and it speaks the exact same protocol as Docker Hub, so `tag`/`push`/`pull` are identical, only the host (`localhost:5000/`) changes.

Three things turn that toy into something usable:

- **Persistence.** Without a volume, blobs vanish when the container dies. Mount one: `-v registry-data:/var/lib/registry` (module 04's named volume, doing exactly its job).
- **TLS + auth.** `registry:2` speaks plain HTTP by default. For anything beyond `localhost`, put it behind a TLS-terminating proxy (Nginx, Caddy) and enable auth (`htpasswd`, OIDC) via its env vars — Docker refuses to talk to a non-localhost registry over plain HTTP unless you explicitly mark it insecure.
- **A real front end.** For production most teams don't run bare `registry:2` — they run **Harbor**, which wraps the *same* server with a UI, RBAC, replication, and built-in scanning.

The takeaway: the registry protocol is simple enough that a private one is a single container, but "production registry" means adding the storage, transport security, access control, and lifecycle policy around it — which is why hosted options and Harbor exist. The next two sections cover two of those operational concerns: caching upstream pulls, and reclaiming space.